# RAG Ingestion Pipeline

In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf

In [ ]:
!pip install sentence-transformers chromadb

In [ ]:
# pypdf => helps to extract data from the pdf 
#pymupdf => helps to process complex pdfs(info from images)
## chromadb => Vector Database
## sentence transformers => chunck embeddings

In [10]:
## In Langchain_community there are different document_loaders like textloader pypdfloader

In [3]:
## Working with the text
from langchain_community.document_loaders import TextLoader


C:\Users\ankur\AppData\Local\Temp\ipykernel_19704\1507986113.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [6]:
loader = TextLoader("python.txt",encoding="utf-8")

In [7]:
document = loader.load()

In [20]:
# Load pdf Data
from langchain_community.document_loaders import PyMuPDFLoader

In [29]:
pdf_loader = PyMuPDFLoader("research1.pdf")

In [30]:
document = pdf_loader.load()

## building pipeline

In [4]:
## Data => Langchain Document
import os
from langchain_community.document_loaders import PyMuPDFLoader

In [5]:
def all_pdf():
    folder_path = "data/pdfs"
    no_of_docs =0
    all_pdfs =[]

    for file in os.listdir(folder_path):
        if file.lower().endswith(".pdf"):
            # Complete pdf path 
            pdf_path = os.path.join(folder_path , file)
            loader = PyMuPDFLoader(pdf_path)
            doc = loader.load()

            all_pdfs.extend(doc)
            no_of_docs +=1
            
    print("Total Pdfs = ",no_of_docs)
    print("Total_pages :",len(all_pdfs))
    return all_pdfs

In [6]:
documents = all_pdf()

Total Pdfs =  2
Total_pages : 36


## Chunk the documents

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [62]:
def chunking(document,chunk_size=1000,chunk_overlap=200):
    text_splitters = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitters.split_documents(document)

    return chunked_docs

In [63]:
chunks = chunking(documents)

In [58]:
type(all_pdfs[0])

langchain_core.documents.base.Document

In [64]:
len(chunks)

195

In [12]:
chunks[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': 'data/pdfs\\research1.pdf', 'file_path': 'data/pdfs\\research1.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu')

In [13]:
from sentence_transformers import SentenceTransformer

In [14]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def generate_embeddings(self, text):
        print("Inside generate_embeddings")
        print("Input type:", type(text))

        embeddings = self.model.encode(text, show_progress_bar=True)

        print("Encoding completed")
        print("Embeddings Shape:", embeddings.shape)

        return embeddings

In [15]:
embeddings_manager = EmbeddingManager()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Initialize our Vector store

In [17]:
import chromadb
import uuid

In [18]:
class VectorStoreManager():
    def __init__(self,persist_directory="data/vector_store",collection_name="pdf_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None ## client who connet with out vector store
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True) ## create directory

        ## Create client
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        ## create collecton
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description":"Vector store for pdf document in RAG"}
        )

        print("Initialized Vector store with the name of :",self.collection_name)
        print("docs in collection count:",self.collection.count())


    def store_embeddings(self,documents,embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Length of documents is not match with the length of embeddings")

        # store => ids , documents , embeddings , metadata

        ids = []
        document_content = []
        embeddings_list = []
        all_metadata = []

        for i , (doc , embd) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            embeddings_list.append(embd.tolist())

            document_content.append(doc.page_content)

            self.collection.add(
                ids=ids,
                documents=document_content,
                metadatas=all_metadata,
                embeddings=embeddings_list
            )

        print("total documents added in the vector store",len(document_content))
        print("docs in collection count:",self.collection.count())
    

In [19]:
vector_store = VectorStoreManager()

Initialized Vector store with the name of : pdf_store
docs in collection count: 338


In [101]:
type(embeddings_manager)

__main__.EmbeddingManager

### store in the vector store

In [65]:
text = [docs.page_content for docs in chunks]

In [66]:
embeddings = embeddings_manager.generate_embeddings(text)

Inside generate_embeddings
Input type: <class 'list'>


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding completed
Embeddings Shape: (195, 384)


In [67]:
vector_store.store_embeddings(chunks , embeddings)

total documents added in the vector store 195
docs in collection count: 871


# Retrieval Pipeline

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class RAGRetriever:
    def __init__(self, embeddings_manager, vector_store):
        self.embeddings_manager = embeddings_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):

        # Query -> Embedding
        query_embedding = self.embeddings_manager.generate_embeddings([query])[0]

        # Semantic Search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        # Check if documents exist
        if results["documents"] and len(results["documents"][0]) > 0:

            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):

                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f" Retrieved {len(retrieved_docs)} documents")

        else:
            print(" No documents found.")

        return retrieved_docs

In [25]:
retriever = RAGRetriever(
    embeddings_manager=embeddings_manager,
    vector_store=vector_store
)

In [26]:
retriever.retrieve("What is RAG ?")

Inside generate_embeddings
Input type: <class 'list'>


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding completed
Embeddings Shape: (1, 384)
 Retrieved 5 documents


[{'id': 'doc_57d96c6a-26d1-4140-ac01-06feef2c6f7a',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'content_length': 288,
   'source': 'data/pdfs\\research2.pdf',
   'page': 0,
   'format': 'PDF 1.5',
   'subject': '',
   'author': '',
   'index': 105,
   'total_pages': 21,
   'modDate': 'D:20240328005445Z',
   'producer': 'pdfTeX-1.40.25',
   'creationDate': 'D:20240328005445Z',
   'moddate': '2024-03-28T00:54:45+00:00',
   'keywords': '',
   'trapped': '',
   'title': '',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'creator': 'LaTeX with hyperref',
   'file_path': 'data/pdfs\\research2.pdf'},
  'distance': 0.46291494369506836,
  'similarity_score': 0.5370850563049316,
  'rank': 1},
 {'id': 'doc_2785db4d

In [27]:
retriever.retrieve("What is encoder and decoder ")

Inside generate_embeddings
Input type: <class 'list'>


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding completed
Embeddings Shape: (1, 384)
 Retrieved 5 documents


[{'id': 'doc_df90988b-b966-4b80-9b31-feecb2c72e9a',
  'document': 'Decoder:\nThe decoder is also composed of a stack of N = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention\nsub-layer in the decoder stack to prevent positions from attending to subsequent positions. This',
  'metadata': {'author': '',
   'page': 2,
   'trapped': '',
   'title': '',
   'file_path': 'data/pdfs\\research1.pdf',
   'producer': 'pdfTeX-1.40.25',
   'creator': 'LaTeX with hyperref',
   'modDate': 'D:20240410211143Z',
   'creationDate': 'D:20240410211143Z',
   'format': 'PDF 1.5',
   'content_length': 495,
   'index': 20,
   'moddate': '2024-04-10T21:11:43+00:00',
   'total_pages': 15,
   'keywords': '',
   'subject': 

## Integrate with LLMs

### GEMINI

In [39]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

In [76]:
llm_gemini = ChatGoogleGenerativeAI(
    google_api_key = GEMINI_KEY,
    model = "gemini-2.5-flash",
    temperature=0.3,
    max_tokens = 5000
)

In [69]:
## Generate augmented output

def generate_output(query , rag_retriever , llm , top_k=5):
    result = rag_retriever.retrieve(query,top_k) ## generate context
    context = "\n".join([doc["document"] for doc in result]) if result else ""

    if not context:
        print("We found no relevant context based on the query")

    # prompt => context + query
    prompt = f"""
                You are an expert AI assistant.

                Use the provided context to answer the question.

                Rules:
                1. Prefer information from the context.
                2. If the context is incomplete, clearly mention that and supplement with accurate general knowledge.
                3. Keep the answer concise and well structured.

                Context:{context}
                        

                    Question:{query}
                                """
    response = llm.invoke(prompt)
    return response.content
    

In [77]:
answer = generate_output("What is rag",retriever,llm_gemini)

Inside generate_embeddings
Input type: <class 'list'>


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding completed
Embeddings Shape: (1, 384)
 Retrieved 5 documents


In [78]:
print(answer)

The provided context discusses various aspects of RAG, including its methods, evolution (e.g., naive RAG), assessment, and integration within Large Language Models (LLMs). However, it does not explicitly define what RAG stands for or its core concept.

Based on general knowledge, RAG stands for **Retrieval-Augmented Generation**. It is a technique used to enhance the capabilities of Large Language Models (LLMs) by allowing them to retrieve relevant information from an external knowledge base before generating a response. This helps LLMs provide more accurate, up-to-date, and contextually relevant answers, reducing hallucinations and grounding their responses in factual data.


## GROQ

In [73]:
from langchain_groq import ChatGroq

In [79]:
llm_groq = ChatGroq(
    groq_api_key = groq_api,
    model = "qwen/qwen3-32b",
    temperature=0.3,
    max_tokens = 5000
)

In [80]:
def generate_output(query , rag_retriever , llm , top_k=5):
    result = rag_retriever.retrieve(query,top_k) ## generate context
    context = "\n".join([doc["document"] for doc in result]) if result else ""

    if not context:
        print("We found no relevant context based on the query")

    # prompt => context + query
    prompt = f"""
                You are an expert AI assistant.

                Use the provided context to answer the question.

                Rules:
                1. Prefer information from the context.
                2. If the context is incomplete, clearly mention that and supplement with accurate general knowledge.
                3. Keep the answer concise and well structured.

                Context:{context}
                        

                    Question:{query}
                                """
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [81]:
output = generate_output("What is encoder and decoder ",retriever,llm_groq)

Inside generate_embeddings
Input type: <class 'list'>


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding completed
Embeddings Shape: (1, 384)
 Retrieved 5 documents


In [82]:
print(output)

<think>
Okay, the user is asking about the encoder and decoder in the context provided. Let me start by reading through the context carefully.

The context mentions that the encoder maps an input sequence of symbols to continuous representations. It's composed of N=6 identical layers, each with two sub-layers: multi-head self-attention and a position-wise fully connected layer. The decoder is also a stack of N=6 layers but has three sub-layers. The third sub-layer in the decoder does multi-head attention over the encoder's output. Also, the decoder's self-attention is modified to prevent attending to subsequent positions, which makes sense for auto-regressive generation.

The user wants a concise explanation of encoder and decoder. I need to structure the answer clearly, using the context first. The encoder's role is to process the input into continuous representations. The decoder generates the output sequence step by step, using the encoder's output and previous symbols. 

Wait, the 